# P01｜像素腦讀論文：LeNet-5 卷積神經網路

論文：Y. LeCun, L. Bottou, Y. Bengio, P. Haffner, **Gradient-Based Learning Applied to Document Recognition**, *Proceedings of the IEEE*, 86(11), 2278–2324, 1998.

⚠️ **這是像素腦照論文自己重建的版本，不是論文作者的原始程式**（論文沒有附官方程式碼）。
架構、層的大小、C3 的連接表、輸入補成 32×32、tanh 的形狀都照論文；
最後的輸出層改成一般的全連接層＋交叉熵，訓練方法也改用常見的 SGD，所以結果會和論文有一點差距。

按上方 **執行階段 → 全部執行** 就能跑。用 CPU 大約要幾分鐘；換成 GPU 會更快（但結果可能差一點點）。

📺 影片：[youtube.com/@pixelbraintw](https://www.youtube.com/@pixelbraintw)

## 0. 準備：載入套件、下載貓咪照片（CC0 授權，作者 AdinaVoicu，Wikimedia Commons）

In [ ]:
import os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from PIL import Image

if not os.path.exists("cat.png"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/pixelbraintw/pixelbrain-code/main/ep02-edges/cat.png", "cat.png")

## 1. MNIST 手寫數字：6 萬張訓練、1 萬張測試

論文把 28×28 的數字放在 32×32 的中間，背景是 -0.1、筆畫是 1.175。

In [ ]:
torch.manual_seed(0)
tf = transforms.Compose([
    transforms.ToTensor(),                          # 筆畫亮、背景暗，0 到 1
    transforms.Pad(2),                              # 28×28 補成 32×32，和論文一樣
    transforms.Lambda(lambda x: x * 1.275 - 0.1),   # 背景 -0.1、筆畫 1.175
])
train = datasets.MNIST("data", train=True, download=True, transform=tf)
test = datasets.MNIST("data", train=False, download=True, transform=tf)
print(len(train), len(test))

## 2. 照論文搭出 LeNet-5

C 是卷積層，S 是子取樣層（論文的 subsampling，現在常叫池化）。C3 只接 S2 的部分特徵圖（論文表 1）。

In [ ]:
def act(x):                       # 論文用的 tanh：1.7159 × tanh(2x/3)
    return 1.7159 * torch.tanh(2 * x / 3)

class Subsample(nn.Module):       # S 層：2×2 四格相加 × 可學的係數 + 偏差
    def __init__(self, ch):
        super().__init__()
        self.w = nn.Parameter(torch.full((ch, 1, 1), 0.25))
        self.b = nn.Parameter(torch.zeros(ch, 1, 1))
    def forward(self, x):
        return F.avg_pool2d(x, 2) * 4 * self.w + self.b

# 論文表 1：C3 的 16 張圖，各自只接 S2 的哪幾張
TABLE = [[0, 1, 2], [1, 2, 3], [2, 3, 4], [3, 4, 5], [4, 5, 0], [5, 0, 1],
         [0, 1, 2, 3], [1, 2, 3, 4], [2, 3, 4, 5], [3, 4, 5, 0], [4, 5, 0, 1], [5, 0, 1, 2],
         [0, 1, 3, 4], [1, 2, 4, 5], [0, 2, 3, 5], [0, 1, 2, 3, 4, 5]]
MASK = torch.zeros(16, 6, 1, 1)
for j, ins in enumerate(TABLE):
    MASK[j, ins] = 1

class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 6, 5)      # 32×32 → 6 張 28×28
        self.s2 = Subsample(6)            # → 6 張 14×14
        self.c3 = nn.Conv2d(6, 16, 5)     # → 16 張 10×10
        self.s4 = Subsample(16)           # → 16 張 5×5
        self.c5 = nn.Conv2d(16, 120, 5)   # → 120 個 1×1
        self.f6 = nn.Linear(120, 84)
        self.out = nn.Linear(84, 10)      # 重建版：論文這裡是 RBF 輸出層
    def forward(self, x):
        x = act(self.s2(act(self.c1(x))))
        x = act(F.conv2d(x, self.c3.weight * MASK, self.c3.bias))
        x = act(self.s4(x))
        x = act(self.c5(x)).flatten(1)
        return self.out(act(self.f6(x)))

## 3. 數參數：論文說大約 6 萬個

C3 只算表 1 裡有接上的權重。最後的輸出層不算在內（論文的 RBF 輸出層是固定的，不用訓練）。

In [ ]:
net = LeNet5()
size = lambda m: sum(p.numel() for p in m.parameters())
n = {"C1": size(net.c1), "S2": size(net.s2), "C3": int(MASK.sum()) * 25 + 16,
     "S4": size(net.s4), "C5": size(net.c5), "F6": size(net.f6)}
print(n)
print("合計", sum(n.values()))

## 4. 訓練 10 輪

每一輪把 6 萬張看過一次，再用沒看過的 1 萬張測試。

In [ ]:
def test_error():
    wrong = 0
    net.eval()
    with torch.no_grad():
        for x, y in torch.utils.data.DataLoader(test, batch_size=1000):
            wrong += (net(x).argmax(1) != y).sum().item()
    net.train()
    return wrong

loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)
opt = torch.optim.SGD(net.parameters(), lr=0.01, momentum=0.9)
for epoch in range(10):
    for x, y in loader:
        loss = F.cross_entropy(net(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    w = test_error()
    print(f"第 {epoch + 1} 輪：錯 {w} 張，錯誤率 {w / 100:.2f}%")

## 5. 看看它認錯哪些

標題是「正確答案 → LeNet-5 的答案」。

In [ ]:
X = torch.stack([test[i][0] for i in range(len(test))])
Y = test.targets
with torch.no_grad():
    pred = net(X).argmax(1)
bad = (pred != Y).nonzero().flatten()
print("認錯", len(bad), "張")
fig, axes = plt.subplots(2, 8, figsize=(12, 3.6))
for ax, i in zip(axes.flat, bad[:16]):
    ax.imshow(X[i, 0], cmap="gray"); ax.axis("off")
    ax.set_title(f"{Y[i].item()} → {pred[i].item()}")
plt.show()

## 6. 第一層學到了什麼？

C1 的 6 個 5×5 卷積核（紅是正、藍是負），沒有人教，是自己學出來的。

In [ ]:
k = net.c1.weight.detach()[:, 0]
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
for ax, w in zip(axes, k):
    ax.imshow(w, cmap="RdBu_r", vmin=-k.abs().max(), vmax=k.abs().max()); ax.axis("off")
plt.show()

## 7. 用 C1 看一隻貓

同一組卷積核滑過貓咪照片，亮的地方就是反應強的地方。

In [ ]:
big = tf(Image.open("cat.png").convert("L").resize((256, 256))).unsqueeze(0)
with torch.no_grad():
    maps = net.c1(big)[0]
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
for ax, m in zip(axes, maps):
    ax.imshow(m.abs(), cmap="gray"); ax.axis("off")
plt.show()

## 8. 咦？把貓丟進去會怎樣

LeNet-5 只認識 0 到 9，不管你給它什麼，它都只能從這 10 個答案裡選一個。

In [ ]:
cat = tf(Image.open("cat.png").convert("L").resize((28, 28))).unsqueeze(0)
with torch.no_grad():
    p = F.softmax(net(cat), 1)[0]
print(f"LeNet-5 覺得這隻貓是 {p.argmax().item()}，信心 {p.max().item():.0%}")

## 🧪 自己玩玩看

1. 把 `Subsample` 換成 `nn.MaxPool2d(2)`、`act` 換成 `F.relu`，看看錯誤率有沒有變好。
2. 把 `range(10)` 改成 `range(20)`（論文大約訓練 20 輪），錯誤率還會再降嗎？
3. 在小畫家寫一個數字，存成圖片上傳，把 `cat.png` 換成你的檔名，看看 LeNet-5 認不認得你的字。

有問題歡迎到影片留言區問像素腦！